# 1. Datos y representaciones

**Responsable: Mónica, autora del cuaderno original.**

Aquí se ha trasladado su preparación de datos, análisis exploratorio,
TF-IDF y Word2Vec. Las funciones reutilizables están en `funciones.py`. Solo Izan modifica ese archivo.
El archivo original sigue conservado fuera de este repositorio.

Abrir desde la carpeta del proyecto. Instalar las dependencias como indica
el README y colocar los dos Excel en `datos/`.
Las salidas se han limpiado: hay que ejecutar para obtener resultados actuales.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer
from wordcloud import WordCloud
from funciones import (
    FIGURAS, RESULTADOS, evaluar, guardar_metricas,
    preprocesar_resumenes, crear_tfidf, crear_word2vec,
    guardar_representacion,
)

FIGURAS.mkdir(parents=True, exist_ok=True)


## Preparación de los datos

Elegimos el primer código con una categoría válida de nivel 2 y guardamos
también su nivel 1. Se usa la misma regla que en el cuaderno original.
Las fechas ausentes se mantienen; su imputación para modelos con metadatos
se debe aprender con entrenamiento.


In [ ]:
from funciones import cargar_datos, preparar_datos, cargar_o_crear_particiones

projects, scivoc = cargar_datos()
df = preparar_datos(projects, scivoc)
conjuntos = cargar_o_crear_particiones(df)
df_train = conjuntos["train"]
df_val = conjuntos["val"]
df_test = conjuntos["test"]

scivoc_clean = scivoc.dropna(subset=["code"]).copy()


In [ ]:
print("INFORMACIÓN DE PROJECTS")
projects.info()

print("\nINFORMACIÓN DE SCIVOC")
scivoc.info()

In [ ]:
missing_values = pd.DataFrame({'Valores ausentes': projects.isnull().sum(), 'Porcentaje (%)': (projects.isnull().sum() / len(projects) * 100).round(2)})
display(missing_values)

In [ ]:
porcentaje = 100 * len(df) / len(projects)
print("Proyectos originales:", len(projects))
print("Proyectos con etiqueta válida:", len(df))
print(f"Porcentaje conservado: {porcentaje:.2f}%")
print("Clases de nivel 1:", df["label_nivel1"].nunique())
print("Clases de nivel 2:", df["label_nivel2"].nunique())
display(df[["projectID", "euroSciVocCode", "label_nivel1", "label_nivel2"]].head())


**Pendiente:** revisar y explicar la diferencia con el 90,31 % aproximado
del enunciado. Si esta revisión cambia proyectos o etiquetas, hay que acordar
el nuevo reparto y repetir los experimentos afectados.


In [ ]:
# Comprobamos cuántos proyectos pertenecen a cada categoría

# Distribución de categorías de nivel 1
distribucion_nivel1 = df['label_nivel1'].value_counts()

# Distribución de categorías de nivel 2
distribucion_nivel2 = df['label_nivel2'].value_counts()

print("Distribución de nivel 1:")
display(distribucion_nivel1.to_frame(name="Número de proyectos"))

print("Top 10 categorías de nivel 2:")
display(distribucion_nivel2.head(10).to_frame(name="Número de proyectos"))

print("Total de categorías de nivel 2:", len(distribucion_nivel2))
print("Total de proyectos:", distribucion_nivel2.sum())

In [ ]:
# Comprobamos las jerarquías

# Comprobamos si el código coincide con el último nivel
inconsistencias = []

for _, row in scivoc_clean.iterrows():
    codigo = int(row['code'])

    niveles = [int(x) for x in str(row['full_code']).split('/') if x]

    if not niveles or codigo != niveles[-1]:
        inconsistencias.append({'code': codigo, 'full_code': row['full_code']})

print("Número de inconsistencias:", len(inconsistencias))
display(pd.DataFrame(inconsistencias).head(10))

## División de los proyectos

Mantenemos el 60 % / 30 % / 10 % y semilla 42 del cuaderno original.
`resultados/particiones.csv` fija el reparto y el orden para los tres cuadernos.


In [ ]:
distribucion = pd.DataFrame({
    'Conjunto': ['Train', 'Validation', 'Test'],
    'Proyectos': [len(df_train), len(df_val), len(df_test)],
    'Porcentaje (%)': [
        round(len(df_train) / len(df) * 100, 2),
        round(len(df_val) / len(df) * 100, 2),
        round(len(df_test) / len(df) * 100, 2)
    ]
})

display(distribucion)

## **`2. Análisis exploratorio de datos (EDA)`**

### **2.1. Distribución de proyectos por categoría de nivel 2**

En la siguiente gráfica podemos observar qué categorías contienen más o menos proyectos.


In [ ]:
# Contamos proyectos por categoría
distribucion = df['label_nivel2'].value_counts().sort_values(ascending=False)

# Representamos la distribución
plt.figure(figsize=(14, 6))

distribucion.plot(kind='bar')

plt.title('Distribución de proyectos por categoría de nivel 2')
plt.xlabel('Categoría EuroSciVoc')
plt.ylabel('Número de proyectos')

plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(FIGURAS / "distribucion_clases.png", dpi=150, bbox_inches="tight")
plt.show()

La distribución permite comprobar el desbalance entre categorías.
Completar la discusión con las cifras de esta ejecución y explicar por
qué se usa macro-F1 además de accuracy.


### **2.2. Clasificador de referencia (baseline)**

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(df_train[["projectID"]], df_train["label_nivel2"])
pred_val = baseline.predict(df_val[["projectID"]])
clases = sorted(df_train["label_nivel2"].unique())
metricas_baseline = evaluar(df_val["label_nivel2"], pred_val, clases=clases)
display(pd.DataFrame([metricas_baseline]))
guardar_metricas(
    "baseline_nivel2", "ninguna", "clase_mayoritaria",
    2, "val", metricas_baseline,
)


Completar la explicación con las métricas anteriores. El test se evalúa
cuando estén cerradas las decisiones de modelado.


### **2.3. Análisis de la longitud de los resúmenes**

In [ ]:
# Calcular el número de palabras de cada resumen
df['longitud_summary'] = df['summary'].str.split().str.len()

# Estadísticas descriptivas
display(df['longitud_summary'].describe().to_frame(name='Número de palabras'))

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(df['longitud_summary'], bins=50, edgecolor='black')

plt.title('Distribución de la longitud de los resúmenes')
plt.xlabel('Número de palabras')
plt.ylabel('Número de proyectos')

plt.tight_layout()
plt.savefig(FIGURAS / "longitud_resumenes.png", dpi=150, bbox_inches="tight")
plt.show()

### **2.4. Distribución temporal de los proyectos**

In [ ]:
# Extraer el año de inicio
df['startYear'] = df['startDate'].dt.year

# Contar proyectos por año
proyectos_anio = df['startYear'].value_counts().sort_index()

display(proyectos_anio.to_frame(name='Número de proyectos'))

In [ ]:
plt.figure(figsize=(10, 5))

proyectos_anio.plot(kind='bar')

plt.title('Distribución de proyectos por año de inicio')
plt.xlabel('Año de inicio')
plt.ylabel('Número de proyectos')

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURAS / "proyectos_por_anio.png", dpi=150, bbox_inches="tight")
plt.show()

### **2.5. Distribución geográfica de los proyectos**

In [ ]:
# Contar proyectos por país coordinador
proyectos_pais = df['coordinatorCountry'].value_counts()

# Mostrar los 10 países con más proyectos
display(proyectos_pais.head(10).to_frame(name='Número de proyectos'))

In [ ]:
# Representamos los 10 países principales
plt.figure(figsize=(10, 5))

proyectos_pais.head(10).plot(kind='bar')

plt.title('Top 10 países coordinadores de proyectos')
plt.xlabel('País')
plt.ylabel('Número de proyectos')

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURAS / "proyectos_por_pais.png", dpi=150, bbox_inches="tight")
plt.show()

### **2.6. Análisis exploratorio del texto**

#### 2.6.1. Palabras más frecuentes

In [ ]:
# Creamos vectorizador eliminando stopwords en inglés
vectorizer = CountVectorizer(stop_words='english', max_features=20)

# Obtenemos frecuencias de palabras
X = vectorizer.fit_transform(df['summary'])

frecuencias = X.sum(axis=0).A1
palabras = vectorizer.get_feature_names_out()

# Crear DataFrame con las palabras más frecuentes
palabras_frecuentes = pd.DataFrame({'Palabra': palabras, 'Frecuencia': frecuencias}).sort_values(by='Frecuencia', ascending=False)

display(palabras_frecuentes)

In [ ]:
plt.figure(figsize=(10, 7))

plt.barh(palabras_frecuentes['Palabra'], palabras_frecuentes['Frecuencia'])

plt.xlabel('Frecuencia')
plt.ylabel('Palabra')
plt.title('20 palabras más frecuentes en los resúmenes')

plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURAS / "palabras_frecuentes.png", dpi=150, bbox_inches="tight")
plt.show()

#### 2.6.2. Nube de palabras

In [ ]:
# Crear diccionario de frecuencias
frecuencias_dict = dict(zip(palabras_frecuentes['Palabra'], palabras_frecuentes['Frecuencia']))

# Generar nube de palabras
wordcloud = WordCloud(width=1000, height=500, background_color='white').generate_from_frequencies(frecuencias_dict)

# Mostrar nube
plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Nube de palabras de los resúmenes')
plt.savefig(FIGURAS / "nube_palabras.png", dpi=150, bbox_inches="tight")
plt.show()

#### 2.6.3. Identificación de categorías científicas

In [ ]:
# Crear diccionario de código a nombre de categoría
nombres_categorias = dict(zip(scivoc_clean['code'].astype(int), scivoc_clean['title']))

# Añadir el nombre de la categoría al DataFrame
df['categoria_nivel2'] = df['label_nivel2'].map(nombres_categorias)

# Comprobar el resultado
display(df[['projectID', 'label_nivel2', 'categoria_nivel2']].head(10))

#### 2.6.4. Ejemplos de proyectos por categorías

In [ ]:
# Seleccionamos las 5 categorías más frecuentes
categorias_top5 = df['label_nivel2'].value_counts().head(5).index

# Obtenemos 2 ejemplos de cada categoría
ejemplos = (df[df['label_nivel2'].isin(categorias_top5)].groupby('label_nivel2', group_keys=False).sample(n=2, random_state=42))

# Mostramos los ejemplos
display(ejemplos[['projectID', 'categoria_nivel2', 'title', 'summary']])

## TF-IDF

Se reutiliza el preprocesamiento del cuaderno original: lematización,
minúsculas, eliminación de stopwords y tokens no alfabéticos o cortos.
El vocabulario se aprende únicamente con train.


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
documentos = {
    nombre: preprocesar_resumenes(parte["summary"], nlp=nlp)
    for nombre, parte in conjuntos.items()
}
print("Original:", df_train["summary"].iloc[0][:500])
print("Limpio:", documentos["train"][0][:500])


In [ ]:
tfidf, matrices_tfidf = crear_tfidf(documentos)
for nombre, matriz in matrices_tfidf.items():
    print(nombre, matriz.shape)
ruta = guardar_representacion("tfidf", matrices_tfidf, conjuntos)
print("Vectores guardados en:", ruta)


## Word2Vec

Se conserva la media de los vectores de palabras conocida del cuaderno inicial.
El entrenamiento utiliza solo train. Se omiten palabras desconocidas y se
asigna un vector de ceros si no queda ninguna palabra utilizable.
Se usa un trabajador y semilla 42 para facilitar la repetición.


In [ ]:
modelo_w2v, matrices_w2v, resumen_oov = crear_word2vec(documentos)
for nombre, matriz in matrices_w2v.items():
    print(nombre, matriz.shape)
display(resumen_oov)
resumen_oov.to_csv(RESULTADOS / "word2vec_oov.csv", index=False)
ruta = guardar_representacion("word2vec", matrices_w2v, conjuntos)
print("Vectores guardados en:", ruta)


## Qué falta de esta parte

- Explicar el porcentaje de proyectos conservados.
- Revisar el análisis y poner sus figuras y comentarios en el README.
- Revisar el efecto de los valores ausentes y de las clases con pocos ejemplos.
- Anotar los parámetros definitivos y las versiones usadas.

Izan puede cargar los vectores desde su cuaderno. Si cambia el preprocesamiento
o algún parámetro de las representaciones, regenerar las cachés.
